# Readme

In [ ]:
wandb:https://wandb.ai/anqiyang00-carnegie-mellon-university/hw3p2-ablations/runs/acfk346p


# Installs

In [1]:
# %pip install torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchtext==0.14.1 torchaudio==0.13.1 torchdata==0.5.1 --extra-index-url https://download.pytorch.org/whl/cu117 -q

In [2]:
# !pip install torchsummaryx==1.3.0
# !pip install wandb --quiet
# !pip install python-Levenshtein -q
# !git clone --recursive https://github.com/parlance/ctcdecode.git
# !pip install wget -q
# %cd ctcdecode
# !pip install . -q
# %cd ..

In [3]:
# !pip install torchsummaryX==1.3.0
# !pip install pandas==1.5.2


This may take a while

# Imports

In [ ]:
import torch
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torchsummaryX import summary
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import torchaudio.transforms as T

import torchaudio.transforms as tat

from sklearn.metrics import accuracy_score
import gc

import zipfile
import pandas as pd
from tqdm import tqdm
import os
import datetime

# imports for decoding and distance calculation
import ctcdecode
import Levenshtein
from ctcdecode import CTCBeamDecoder

import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
print("Device: ", device)

# Kaggle Setup

In [5]:
import os
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
#kaggle account and API
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w+") as f:
    f.write('{"username":"angelanqiya","key":"9b813d342c3b556cafd2689211d153ef"}')


os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)



In [6]:

# !kaggle competitions download -c 11-785-hw3p2-f24

In [7]:
'''
This will take a couple minutes, but you should see at least the following:
11-785-f24-hw3p2  ctcdecode  hw3p2asr-f24.zip  sample_data
'''
# !unzip -q 11-785-hw3p2-f24.zip
!ls

 11785-f24-hw3p2	     'HW3P2_Starter_F24 copy.ipynb'
 11-785-hw3p2-f24.zip	      HW3P2_Starter_F24.ipynb
 1embed_2lstm_2pblstm.ipynb   random_submission.csv
 2embed_1lstm_2pblstm.ipynb   requirements.txt
'2lstm_3pblstm copy.ipynb'    sample_submission.csv
 2lstm_3pblstm.ipynb	      submission.csv
 checkpoint		      wandb
 ctcdecode


# Google Drive

In [8]:
# from google.colab import drive
# drive.mount('/content/gdrive')

# Dataset and Dataloader

In [9]:
# ARPABET PHONEME MAPPING
# DO NOT CHANGE

CMUdict_ARPAbet = {
    "" : " ",
    "[SIL]": "-", "NG": "G", "F" : "f", "M" : "m", "AE": "@",
    "R"    : "r", "UW": "u", "N" : "n", "IY": "i", "AW": "W",
    "V"    : "v", "UH": "U", "OW": "o", "AA": "a", "ER": "R",
    "HH"   : "h", "Z" : "z", "K" : "k", "CH": "C", "W" : "w",
    "EY"   : "e", "ZH": "Z", "T" : "t", "EH": "E", "Y" : "y",
    "AH"   : "A", "B" : "b", "P" : "p", "TH": "T", "DH": "D",
    "AO"   : "c", "G" : "g", "L" : "l", "JH": "j", "OY": "O",
    "SH"   : "S", "D" : "d", "AY": "Y", "S" : "s", "IH": "I",
    "[SOS]": "[SOS]", "[EOS]": "[EOS]"
}

CMUdict = list(CMUdict_ARPAbet.keys())
ARPAbet = list(CMUdict_ARPAbet.values())


PHONEMES = CMUdict[:-2]
LABELS = ARPAbet[:-2]



In [10]:
# You might want to play around with the mapping as a sanity check here
print(f"Length of PHONEMES: {len(PHONEMES)}")
print(f"Length of LABELS: {len(LABELS)}")

print("Sample phoneme-label mappings:")
for i in range(5):  # Adjust the range as needed
    print(f"{PHONEMES[i]} -> {LABELS[i]}")
    



Length of PHONEMES: 41
Length of LABELS: 41
Sample phoneme-label mappings:
 ->  
[SIL] -> -
NG -> G
F -> f
M -> m


### Train Data

In [11]:
class AudioDataset(torch.utils.data.Dataset):

    # For this homework, we give you full flexibility to design your data set class.
    # Hint: The data from HW1 is very similar to this HW

    #TODO
    def __init__(self,root, phonemes = PHONEMES, context=20, partition= "train-clean-100"):
        '''
        Initializes the dataset.

        INPUTS: What inputs do you need here?
        '''
        self.root = root
        self.partition = partition
        self.context = context
        # Load the directory and all files in them

        self.mfcc_dir = os.path.join(root,partition,'mfcc')
        self.transcript_dir = os.path.join(root,partition,'transcript')

        self.mfcc_files = sorted(os.listdir(self.mfcc_dir))
        self.transcript_files = sorted(os.listdir(self.transcript_dir))

        self.PHONEMES = PHONEMES
        self.phonemes_ind ={phoneme: index for index, phoneme in enumerate(self.PHONEMES)}

        #TODO
        # WHAT SHOULD THE LENGTH OF THE DATASET BE?
        self.length = len(self.mfcc_files)
        
        self.mfccs, self.transcripts = [], []
        for i in range(len(self.mfcc_files)):
        #   Load a single mfcc
            mfcc  = np.load(os.path.join(self.mfcc_dir,self.mfcc_files[i]))
            #   Do Cepstral Normalization of mfcc (explained in writeup)
            mfcc = (mfcc - np.mean(mfcc,axis =0,keepdims=True))/(np.std(mfcc,axis= 0,keepdims=True))
            self.mfccs.append(mfcc)
                
            #load transcript
            transcript = np.load(os.path.join(self.transcript_dir,self.transcript_files[i]))
            # Convert to indices (do i need to remove eos and sos?)
            transcript = transcript[1:-1]
            transcript = [self.PHONEMES.index(p) for p in transcript]
            #append transcripts
            self.transcripts.append(transcript)
            
       
        


    def __len__(self):

        '''
        TODO: What do we return here?
        '''
        return self.length
        
        # raise NotImplemented

    def __getitem__(self, ind):
        '''
        TODO: RETURN THE MFCC COEFFICIENTS AND ITS CORRESPONDING LABELS

        If you didn't do the loading and processing of the data in __init__,
        do that here.

        Once done, return a tuple of features and labels.
        '''
        # Load a single mfcc
        mfcc  = self.mfccs[ind]
        transcript = self.transcripts[ind]
        mfcc      = torch.FloatTensor(mfcc) # Convert to tensors
        phonemes    = torch.tensor(transcript)
        return mfcc, phonemes
        


    def collate_fn(self,batch):
        '''
        TODO:
        1.  Extract the features and labels from 'batch'
        2.  We will additionally need to pad both features and labels,
            look at pytorch's docs for pad_sequence
        3.  This is a good place to perform transforms, if you so wish.
            Performing them on batches will speed the process up a bit.
        4.  Return batch of features, labels, lenghts of features,
            and lengths of labels.
        '''
        # batch of input mfcc coefficients
        batch_mfcc , batch_transcript = zip(*batch)
        # Convert lists to tensors and pad sequences
        batch_mfcc = [torch.tensor(mfcc, dtype=torch.float) for mfcc in batch_mfcc]
        batch_transcript = [torch.tensor(transcript, dtype=torch.long) for transcript in batch_transcript]


        # HINT: CHECK OUT -> pad_sequence (imported above)
        # Also be sure to check the input format (batch_first)
        batch_mfcc_pad = pad_sequence(batch_mfcc, batch_first=True)
        lengths_mfcc = [len(mfcc) for mfcc in batch_mfcc]
        # print(lengths_mfcc)

        batch_transcript_pad = pad_sequence(batch_transcript, batch_first=True)
        lengths_transcript = [len(transcript) for transcript in batch_transcript]

        # You may apply some transformation, Time and Frequency masking, here in the collate function;
        # Food for thought -> Why are we applying the transformation here and not in the __getitem__?
        #                  -> Would we apply transformation on the validation set as well?
        #                  -> Is the order of axes / dimensions as expected for the transform functions?

        # Return the following values: padded features, padded labels, actual length of features, actual length of the labels
        
        # time_mask = T.TimeMasking(time_mask_param=30)
        # freq_mask = T.FrequencyMasking(freq_mask_param=15)

        # Apply time and frequency masking
        # batch_mfcc_pad = time_mask(batch_mfcc_pad)
        # batch_mfcc_pad = freq_mask(batch_mfcc_pad)
        return batch_mfcc_pad, batch_transcript_pad, torch.tensor(lengths_mfcc), torch.tensor(lengths_transcript)



In [12]:
# Dataset class to load train and validation data

class AudioVALDataset(torch.utils.data.Dataset):

    def __init__(self,root, phonemes = PHONEMES, context=20, partition= "dev-clean"):
        '''
        Initializes the dataset.

        INPUTS: What inputs do you need here?
        '''
        self.root = root
        self.partition = partition
        self.context = context
        # Load the directory and all files in them

        self.mfcc_dir = os.path.join(root,partition,'mfcc')
        self.transcript_dir = os.path.join(root,partition,'transcript')

        self.mfcc_files = sorted(os.listdir(self.mfcc_dir))
        self.transcript_files = sorted(os.listdir(self.transcript_dir))

        self.PHONEMES = PHONEMES
        self.phonemes_ind ={phoneme: index for index, phoneme in enumerate(self.PHONEMES)}

        #TODO
        # WHAT SHOULD THE LENGTH OF THE DATASET BE?
        self.length = len(self.mfcc_files)
        
        self.mfccs, self.transcripts = [], []
        for i in range(len(self.mfcc_files)):
        #   Load a single mfcc
            mfcc  = np.load(os.path.join(self.mfcc_dir,self.mfcc_files[i]))
            #   Do Cepstral Normalization of mfcc (explained in writeup)
            mfcc = (mfcc - np.mean(mfcc,axis =0, keepdims=True))/(np.std(mfcc,axis= 0, keepdims=True))
            self.mfccs.append(mfcc)
                
            #load transcript
            transcript = np.load(os.path.join(self.transcript_dir,self.transcript_files[i]))
            # Convert to indices (do i need to remove eos and sos?)
            transcript = transcript[1:-1]
            transcript = [self.PHONEMES.index(p) for p in transcript]
            #append transcripts
            self.transcripts.append(transcript)
            
       
        


    def __len__(self):

        '''
        TODO: What do we return here?
        '''
        return self.length
        
        # raise NotImplemented

    def __getitem__(self, ind):
        '''
        TODO: RETURN THE MFCC COEFFICIENTS AND ITS CORRESPONDING LABELS

        If you didn't do the loading and processing of the data in __init__,
        do that here.

        Once done, return a tuple of features and labels.
        '''
        # Load a single mfcc
        mfcc  = self.mfccs[ind]
        transcript = self.transcripts[ind]
        mfcc      = torch.FloatTensor(mfcc) # Convert to tensors
        phonemes    = torch.tensor(transcript)
        return mfcc, phonemes
        


    def collate_fn(self,batch):
        '''
        TODO:
        1.  Extract the features and labels from 'batch'
        2.  We will additionally need to pad both features and labels,
            look at pytorch's docs for pad_sequencelog_probs_input
        3.  This is a good place to perform transforms, if you so wish.
            Performing them on batches will speed the process up a bit.
        4.  Return batch of features, labels, lenghts of features,
            and lengths of labels.
        '''
        # batch of input mfcc coefficients
        batch_mfcc , batch_transcript = zip(*batch)
        # Convert lists to tensors and pad sequences
        batch_mfcc = [torch.tensor(mfcc, dtype=torch.float) for mfcc in batch_mfcc]
        batch_transcript = [torch.tensor(transcript, dtype=torch.long) for transcript in batch_transcript]


        # HINT: CHECK OUT -> pad_sequence (imported above)
        # Also be sure to check the input format (batch_first)
        batch_mfcc_pad = pad_sequence(batch_mfcc, batch_first=True)
        lengths_mfcc = [len(mfcc) for mfcc in batch_mfcc]
        # print(lengths_mfcc)
        batch_transcript_pad = pad_sequence(batch_transcript, batch_first=True)
        lengths_transcript = [len(transcript) for transcript in batch_transcript]

        # You may apply some transformation, Time and Frequency masking, here in the collate function;
        # Food for thought -> Why are we applying the transformation here and not in the __getitem__?
        #                  -> Would we apply transformation on the validation set as well?
        #                  -> Is the order of axes / dimensions as expected for the transform functions?

        # Return the following values: padded features, padded labels, actual length of features, actual length of the labels
        
        # time_mask = T.TimeMasking(time_mask_param=30)
        # freq_mask = T.FrequencyMasking(freq_mask_param=15)

        # Apply time and frequency masking
        # batch_mfcc_pad = time_mask(batch_mfcc_pad)
        # batch_mfcc_pad = freq_mask(batch_mfcc_pad)
        return batch_mfcc_pad, batch_transcript_pad, torch.tensor(lengths_mfcc), torch.tensor(lengths_transcript)



### Test Data

In [13]:
# Test Dataloader
#TODO
class AudioDatasetTest(torch.utils.data.Dataset):
    def __init__(self,root, PHONEMES = PHONEMES, partition= "test-clean"):
        '''
        Initializes the dataset.

        INPUTS: What inputs do you need here?
        '''
        self.root = root
        self.partition = partition
        # Load the directory and all files in them

        self.mfcc_dir = os.path.join(root,partition,'mfcc')
       

        self.mfcc_files = sorted(os.listdir(self.mfcc_dir))
       

        self.PHONEMES = PHONEMES
        

        #TODO
        # WHAT SHOULD THE LENGTH OF THE DATASET BE?
        self.length = len(self.mfcc_files)
        
        self.mfccs = []
        for i in range(len(self.mfcc_files)):
        #   Load a single mfcc
            mfcc  = np.load(os.path.join(self.mfcc_dir,self.mfcc_files[i]))
            #   Do Cepstral Normalization of mfcc (explained in writeup)
            mfcc = (mfcc - np.mean(mfcc,axis =0,keepdims=True))/(np.std(mfcc,axis= 0,keepdims=True))
            self.mfccs.append(mfcc)

    def __len__(self):

        '''
        TODO: What do we return here?
        '''
        return self.length
        
        # raise NotImplemented

    def __getitem__(self, ind):
        '''
        TODO: RETURN THE MFCC COEFFICIENTS AND ITS CORRESPONDING LABELS

        If you didn't do the loading and processing of the data in __init__,
        do that here.

        Once done, return a tuple of features and labels.
        '''
        # Load a single mfcc
        mfcc  = self.mfccs[ind]
        
        mfcc      = torch.FloatTensor(mfcc) # Convert to tensors
    
        return mfcc
        


    def collate_fn(self,batch):
        '''
        TODO:
        1.  Extract the features and labels from 'batch'
        2.  We will additionally need to pad both features and labels,
            look at pytorch's docs for pad_sequence
        3.  This is a good place to perform transforms, if you so wish.
            Performing them on batches will speed the process up a bit.
        4.  Return batch of features, labels, lenghts of features,
            and lengths of labels.
        '''
        # batch of input mfcc coefficients
        batch_mfcc  = batch
        # Convert lists to tensors and pad sequences
        batch_mfcc = [torch.tensor(mfcc, dtype=torch.float) for mfcc in batch_mfcc]
        # HINT: CHECK OUT -> pad_sequence (imported above)
        # Also be sure to check the input format (batch_first)
        batch_mfcc_pad = pad_sequence(batch_mfcc, batch_first=True)
        lengths_mfcc = [len(mfcc) for mfcc in batch_mfcc]
        return batch_mfcc_pad, torch.tensor(lengths_mfcc)

### Config - Hyperparameters

In [ ]:

config = {
    "root": './11-785-f24-hw3p2',
    "lr"         : 2e-3,
    "epochs"     : 100,
    "batch_size" : 256,
    'out_size' :28,
    "embed_size" : 256,
    'architecture'  : '1embeding-1blstm-2pblstm',
    #model
    'dropout'       : 0.3,
    'beam_width'    : 3,
    'weight_decay'  : 1e-3,
    "factor"  : 0.5

}

# You may pass this as a parameter to the dataset class above
# This will help modularize your implementation
transforms = [] # set of tranformations

### Data loaders

In [15]:
# get me RAMMM!!!!
import gc
gc.collect()

0

In [16]:
# Create objects for the dataset class
#TODO: Create a dataset object using the AudioDataset class for the training data
train_data = AudioDataset(root = config['root'], phonemes = PHONEMES, partition= "train-clean-100")
val_data = AudioVALDataset(root = config['root'], phonemes = PHONEMES,partition= "dev-clean")


# TODO: Create a dataset object using the AudioTestDataset class for the test data
test_data = AudioDatasetTest(root = config['root'], partition= "test-clean")


# Define dataloaders for train, val and test datasets
# Dataloaders will yield a batch of frames and phonemes of given batch_size at every iteration
# We shuffle train dataloader but not val & test dataloader. Why?

train_loader = torch.utils.data.DataLoader(
    dataset     = train_data,
    num_workers = 4,
    batch_size  = config['batch_size'],
    pin_memory  = True,
    shuffle     = True,
    collate_fn = train_data.collate_fn
)

val_loader = torch.utils.data.DataLoader(
    dataset     = val_data,
    num_workers = 2,
    batch_size  = config['batch_size'],
    pin_memory  = True,
    shuffle     = False,
    collate_fn = val_data.collate_fn
)

test_loader = torch.utils.data.DataLoader(
    dataset     = test_data,
    num_workers = 2,
    batch_size  = config['batch_size'],
    pin_memory  = True,
    shuffle     = False,
    collate_fn = test_data.collate_fn
)


print("Batch size     : ", config['batch_size'])

print("Output symbols : ", len(PHONEMES))

print("Train dataset samples = {}, batches = {}".format(train_data.__len__(), len(train_loader)))
print("Validation dataset samples = {}, batches = {}".format(val_data.__len__(), len(val_loader)))
print("Test dataset samples = {}, batches = {}".format(test_data.__len__(), len(test_loader)))

Batch size     :  256
Output symbols :  41
Train dataset samples = 28539, batches = 112
Validation dataset samples = 2703, batches = 11
Test dataset samples = 2620, batches = 11


In [17]:
# sanity check
for data in train_loader:
    x, y, lx, ly = data
    print(x.shape, y.shape, lx.shape, ly.shape)
    break

torch.Size([256, 1698, 28]) torch.Size([256, 208]) torch.Size([256]) torch.Size([256])


# NETWORK

## Basic

This is a basic block for understanding, you can skip this and move to pBLSTM one

In [18]:
# torch.cuda.empty_cache()

# class Network(nn.Module):

#     def __init__(self):

#         super(Network, self).__init__()

#         # Adding some sort of embedding layer or feature extractor might help performance.
#         # self.embedding = ?

#         # TODO : look up the documentation. You might need to pass some additional parameters.
#         self.lstm = nn.LSTM(input_size = __, hidden_size = 256, num_layers = 1)

#         self.classification = nn.Sequential(
#             #TODO: Linear layer with in_features from the lstm module above and out_features = OUT_SIZE
#         )


#         self.logSoftmax = #TODO: Apply a log softmax here. Which dimension would apply it on ?

#     def forward(self, x, lx):
#         #TODO
#         # The forward function takes 2 parameter inputs here. Why?
#         # Refer to the handout for hints
#         pass

## Initialize Basic Network
(If trying out the basic Network)

In [19]:
# torch.cuda.empty_cache()

# model = Network().to(device)
# summary(model, x.to(device), lx) # x and lx come from the sanity check above :)

## ASR Network

### Pyramid Bi-LSTM (pBLSTM)

In [20]:
# Utils for network
torch.cuda.empty_cache()

class PermuteBlock(torch.nn.Module):
    def forward(self, x):
        return x.transpose(1, 2)

In [21]:
class pBLSTM(torch.nn.Module):

    '''
    Pyramidal BiLSTM
    Read the write up/paper and understand the concepts and then write your implementation here.

    At each step,
    1. Pad your input if it is packed (Unpack it)
    2. Reduce the input length dimension by concatenating feature dimension
        (Tip: Write down the shapes and understand)
        (i) How should  you deal with odd/even length input?
        (ii) How should you deal with input length array (x_lens) after truncating the input?
    3. Pack your input
    4. Pass it into LSTM layer

    To make our implementation modular, we pass 1 layer at a time.
    '''

    def __init__(self, input_size, hidden_size):
        super(pBLSTM, self).__init__()

        self.blstm = nn.LSTM(input_size=2*input_size, hidden_size=hidden_size, num_layers=2, dropout= config['dropout'],bidirectional=True,batch_first=True)

    def forward(self, x_packed): # x_packed is a PackedSequence
        # print('x_packed_inputforlstm:', x_packed.data.shape)
        # TODO: Pad Packed Sequence
        x, lens_unpacked = pad_packed_sequence(x_packed, batch_first = True)
        # Call self.trunc_reshape() which downsamples the time steps of x and increases the feature dimensions as mentioned above
        # self.trunc_reshape will return 2 outputs. What are they? Think about what quantites are changing.
        x, x_lens= self.trunc_reshape(x,lens_unpacked)
        
        # TODO: Pack Padded Sequence. What output(s) would you get?
        x = pack_padded_sequence(x, x_lens,batch_first=True,enforce_sorted=False)
        # TODO: Pass the sequence through bLSTM
        # print('x_packed_inputforblstm:', x.data.shape)
        x, hidden = self.blstm(x)
        # What do you return?

        return x

    def trunc_reshape(self, x, x_lens):
        # TODO: If you have odd number of timesteps, how can you handle it? (Hint: You can exclude them)
        # TODO: Reshape x. When reshaping x, you have to reduce number of timesteps by a downsampling factor while increasing number of features by the same factor
        # TODO: Reduce lengths by the same downsampling factor
        if x.shape[1]%2 != 0:
            x = x[:,:-1,:]
            
        # Reshape x
        # Reshape to downsample time dimension by 2
        batch_size, seq_len, feature_size = x.size()
        x = x.view(batch_size, seq_len // 2, feature_size * 2)  # Combine pairs of time steps

        # Adjust the lengths accordingly
        lens = x_lens // 2

        return x, lens

### Encoder

In [22]:
class LockedDropout(nn.Module):
    def __init__(self, drop_prob):
        super(LockedDropout, self).__init__()
        self.prob = drop_prob
    def forward(self, x):
        if not self.training or not self.prob: # turn it off during inference
            return x
        x, x_lens = pad_packed_sequence(x, batch_first = True)
        m = x.new_empty(x.size(0), 1, x.size(2),requires_grad=False).bernoulli_(1 - self.prob)
        mask = m / (1 - self.prob)
        mask = mask.expand_as(x)
        out = x * mask
        out = pack_padded_sequence(out,x_lens, batch_first = True, enforce_sorted= False)
        return out

In [23]:
class Encoder(torch.nn.Module):
    '''
    The Encoder takes utterances as inputs and returns latent feature representations
    '''
    def __init__(self, input_size, encoder_hidden_size):
        super(Encoder, self).__init__()


        #self.embedding = #TODO: You can use CNNs as Embedding layer to extract features. Keep in mind the Input dimensions and expected dimension of Pytorch CNN.
        # self.embedding = DCNN(stride = 1. input_size = input_size, out_channels = input_size)
        self.embedding = nn.Sequential(
            PermuteBlock(),
            # out_channel= input_size
            nn.Conv1d(input_size, out_channels= config['out_size'], kernel_size=5, stride = 1,padding = 2),
            # nn.GroupNorm(num_groups = 1, num_channels = in_features), # Layer norm
            nn.BatchNorm1d(num_features = config['out_size']),
            nn.GELU(),
            PermuteBlock()
            # nn.Linear(expansion*in_features, out_features, bias =True) # shrink
        )
        #self.pBLSTMs = torch.nn.Sequential( # How many pBLSTMs are required?
            # TODO: Fill this up with pBLSTMs - What should the input_size be?
            # Hint: You are downsampling timesteps by a factor of 2, upsampling features by a factor of 2 and the LSTM is bidirectional)
            # Optional: Dropout/Locked Dropout after each pBLSTM (Not needed for early submission)
            # https://github.com/salesforce/awd-lstm-lm/blob/dfd3cb0235d2caf2847a4d53e1cbd495b781b5d2/locked_dropout.py#L5
            # ...
            # ...
        self.lstm1 = torch.nn.LSTM(input_size=config['out_size'], hidden_size= encoder_hidden_size, num_layers=4, dropout= config['dropout'],bidirectional=True,batch_first=True)
        # self.lstm2 = torch.nn.LSTM(input_size=encoder_hidden_size*2, hidden_size= encoder_hidden_size, num_layers=1, dropout= config['dropout'],bidirectional=True,batch_first=True)
        
        self.pBLSTMs = torch.nn.Sequential(pBLSTM(input_size=encoder_hidden_size*2, hidden_size= encoder_hidden_size),
                                            LockedDropout(0.3),
                                            pBLSTM(encoder_hidden_size*2, encoder_hidden_size),
                                            LockedDropout(0.3),
                                      )
                                             
                                           

    def forward(self, x, x_lens):
        # Where are x and x_lens coming from? The dataloader
        # print('Encoder Input:', x.shape)
        # print('Encoder Input:', x_lens)
        
        #TODO: Call the embedding layer
        embed_layer = self.embedding(x)
        #print(embed_layer.shape)
        clamped_lx = x_lens.clamp(max=embed_layer.shape[1]) # to match the embedding
        # TODO: Pack Padded Sequence
        x = pack_padded_sequence(embed_layer, clamped_lx,batch_first=True,enforce_sorted=False)
        # print('Pack Padded Sequence:', x.data.shape)
        # TODO: Pass Sequence through the pyramidal Bi-LSTM layer
        x_lstm1, _ = self.lstm1(x)
        # x_lstm2, _ = self.lstm2(x_lstm1)
       
        x = self.pBLSTMs(x_lstm1)
        # TODO: Pad Packed Sequence
        encoder_outputs, encoder_lens = pad_packed_sequence(x, batch_first=True)


        # Remember the number of output(s) each function returns

        return encoder_outputs, encoder_lens

### Decoder

In [24]:
class Decoder(torch.nn.Module):

    def __init__(self, embed_size, output_size= 41):
        super().__init__()
        #cylinder
        self.mlp = torch.nn.Sequential(
            PermuteBlock(), torch.nn.BatchNorm1d(embed_size), PermuteBlock(),
            torch.nn.Linear(embed_size, 512), torch.nn.GELU(),
            PermuteBlock(), torch.nn.BatchNorm1d(512), PermuteBlock(),
            torch.nn.Dropout(0.2),
            # torch.nn.Linear(1024, 2048), torch.nn.GELU(),
            # PermuteBlock(), torch.nn.BatchNorm1d(2048), PermuteBlock(),
            # torch.nn.Dropout(0.2),
            torch.nn.Linear(512, 256), torch.nn.GELU(),
            PermuteBlock(), torch.nn.BatchNorm1d(256), PermuteBlock(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(256, 128), torch.nn.GELU(),
            PermuteBlock(), torch.nn.BatchNorm1d(128), PermuteBlock(),
            torch.nn.Dropout(0.1),
            torch.nn.Linear(128, output_size),
    
            
            
            #Use Permute Block before and after BatchNorm1d() to match the size
        )

        self.softmax = torch.nn.LogSoftmax(dim=2)

    def forward(self, encoder_out):
        #TODO call your MLP
        #TODO Think what should be the final output of the decoder for the classification
        out = self.mlp(encoder_out)
        out = self.softmax(out)
        return out

In [25]:
class ASRModel(torch.nn.Module):

    def __init__(self, input_size, embed_size= config['embed_size'], output_size= len(PHONEMES)):
        super().__init__()

        # self.augmentations  = torch.nn.Sequential(
        #     #TODO Add Time Masking/ Frequency Masking
        #     #Hint: See how to use PermuteBlock() function defined above
        #     PermuteBlock(),
        #     T.TimeMasking(time_mask_param=30),
        #     T.FrequencyMasking(freq_mask_param=15),
        #     PermuteBlock()
        # )
        self.encoder        = Encoder(input_size, embed_size)
        self.decoder        = Decoder(embed_size*2, output_size)



    def forward(self, x, lengths_x):

        # if self.training:
        #     x = self.augmentations(x)

        encoder_out, encoder_lens   = self.encoder(x, lengths_x)
        decoder_out                 = self.decoder(encoder_out)

        return decoder_out, encoder_lens

## Initialize ASR Network

In [26]:
from torchsummaryX import summary

In [27]:
model = ASRModel(
    input_size  = 28,
    embed_size  = config['embed_size'],
    output_size = len(PHONEMES)
).to(device)
print(model)
# summary(model, x.to(device), lx)

ASRModel(
  (encoder): Encoder(
    (embedding): Sequential(
      (0): PermuteBlock()
      (1): Conv1d(28, 28, kernel_size=(5,), stride=(1,), padding=(2,))
      (2): BatchNorm1d(28, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): GELU(approximate='none')
      (4): PermuteBlock()
    )
    (lstm1): LSTM(28, 256, num_layers=4, batch_first=True, dropout=0.3, bidirectional=True)
    (pBLSTMs): Sequential(
      (0): pBLSTM(
        (blstm): LSTM(1024, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
      )
      (1): LockedDropout()
      (2): pBLSTM(
        (blstm): LSTM(1024, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
      )
      (3): LockedDropout()
    )
  )
  (decoder): Decoder(
    (mlp): Sequential(
      (0): PermuteBlock()
      (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): PermuteBlock()
      (3): Linear(in_features=512, out_features=512, bias=Tru

# Training Config
Initialize Loss Criterion, Optimizer, CTC Beam Decoder, Scheduler, Scaler (Mixed-Precision), etc.

In [28]:
#TODO


# Define CTC loss as the criterion.
criterion =torch.nn.CTCLoss(blank=0, reduction='mean',zero_infinity=True).to(device)

# CTC Loss: https://pytorch.org/docs/stable/generated/torch.nn.CTCLoss.html
# Refer to the handout for hints

optimizer =  torch.optim.AdamW(model.parameters(), lr= config['lr'],weight_decay = config['weight_decay'])
# optimizer =torch.optim.Adam(model.parameters(), lr= config['lr'])


# Declare the decoder. Use the CTC Beam Decoder to decode phonemes
# CTC Beam Decoder Doc: https://github.com/parlance/ctcdecode
decoder = CTCBeamDecoder(
    LABELS,
    model_path=None,
    alpha=0,
    beta=0,
    cutoff_top_n=40,
    cutoff_prob=1.0,
    beam_width=config['beam_width'],
    num_processes=4,
    blank_id=0,
    log_probs_input=True
)


# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode = 'min', patience = 3,factor =config['factor'] , threshold=1e-2)

# Mixed Precision, if you need it
scaler = torch.cuda.amp.GradScaler()

# Decode Prediction

In [29]:
def decode_prediction(output, output_lens, decoder, PHONEME_MAP= LABELS):

    # TODO: look at docs for CTC.decoder and find out what is returned here. Check the shape of output and expected shape in decode.
    # (...) = decoder.decode(output, seq_lens= output_lens) #lengths - list of lengths
    beam_results, beam_scores, timesteps, out_lens = decoder.decode(output, seq_lens= output_lens)

    pred_strings                    = []

    for i in range(output_lens.shape[0]):
        #TODO: Create the prediction from the output of decoder.decode. Don't forget to map it using PHONEMES_MAP.
        beam_results_string = ''.join([PHONEME_MAP[i] for i in beam_results[i][0][:out_lens[i][0]]])
        pred_strings.append(beam_results_string)
    return pred_strings

def calculate_levenshtein(output, label, output_lens, label_lens, decoder, PHONEME_MAP= LABELS): # y - sequence of integers
    # print('output:', output.shape)
    # print('label:', label.shape)
    dist            = 0
    batch_size      = label.shape[0]

    pred_strings    = decode_prediction(output, output_lens, decoder, PHONEME_MAP)

    for i in range(batch_size):
        # TODO: Get predicted string and label string for each element in the batch
        pred_string = pred_strings[i]
        label_string = ''.join([PHONEME_MAP[i] for i in label[i][:label_lens[i]]])
        dist += Levenshtein.distance(pred_string, label_string)

    dist /= batch_size 
    # raise NotImplemented
    return dist

# Test Implementation

In [30]:
torch.cuda.empty_cache()

In [31]:
torch.cuda.empty_cache()
model.eval()
for i, data in enumerate(val_loader, 0):
    x, y, lx, ly = data
    x, y = x.to(device), y.to(device)
    h, lh = model(x, lx)
    print(h.shape)
    print(calculate_levenshtein(h, y, lx, ly, decoder, LABELS))

    h = torch.permute(h, (1, 0, 2))
    print(h.shape, y.shape)
    loss = criterion(h, y, lh, ly)
    print(loss)

    # print(calculate_levenshtein(h, y, lx, ly, decoder, LABELS))

    break

torch.Size([256, 734, 41])
208.609375
torch.Size([734, 256, 41]) torch.Size([256, 265])
tensor(7.1000, device='cuda:2', grad_fn=<MeanBackward0>)


# WandB

You will need to fetch your api key from wandb.ai

In [32]:
import wandb
wandb.login(key="48892fbec146146324ae6560bf2f084223ecdeb6") #API Key is in your wandb account, under settings (wandb.ai/settings)

wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: anqiyang00 (anqiyang00-carnegie-mellon-university). Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/ljc/.netrc


True

In [33]:
run = wandb.init(
    name = "submission", ## Wandb creates random run names if you skip this field
    reinit = True, ### Allows reinitalizing runs when you re-run this cell
    # run_id = ### Insert specific run id here if you want to resume a previous run
    # resume = "must" ### You need this to resume previous runs, but comment out reinit = True when using this
    project = "hw3p2-ablations", ### Project should be created in your wandb account
    config = config ### Wandb Config for your run
)

# Train Functions

In [34]:
from tqdm import tqdm

def train_model(model, train_loader, criterion, optimizer):

    model.train()
    batch_bar = tqdm(total=len(train_loader), dynamic_ncols=True, leave=False, position=0, desc='Train')

    total_loss = 0

    for i, data in enumerate(train_loader):
        optimizer.zero_grad()

        x, y, lx, ly = data
        x, y = x.to(device), y.to(device)

        with torch.cuda.amp.autocast():
            h, lh = model(x, lx)
            h = torch.permute(h, (1, 0, 2))
            loss = criterion(h, y, lh, ly)

        total_loss += loss.item()

        batch_bar.set_postfix(
            loss="{:.04f}".format(float(total_loss / (i + 1))),
            lr="{:.06f}".format(float(optimizer.param_groups[0]['lr'])))

        batch_bar.update() # Update tqdm bar

        # Another couple things you need for FP16.
        scaler.scale(loss).backward() # This is a replacement for loss.backward()
        scaler.step(optimizer) # This is a replacement for optimizer.step()
        scaler.update() # This is something added just for FP16

        del x, y, lx, ly, h, lh, loss
        torch.cuda.empty_cache()

    batch_bar.close() # You need this to close the tqdm bar

    return total_loss / len(train_loader)


def validate_model(model, val_loader,decoder,phoneme_map= LABELS):

    model.eval()
    batch_bar = tqdm(total=len(val_loader), dynamic_ncols=True, position=0, leave=False, desc='Val')

    total_loss = 0
    vdist = 0

    for i, data in enumerate(val_loader):

        x, y, lx, ly = data
        x, y = x.to(device), y.to(device)

        with torch.inference_mode():
            h, lh = model(x, lx)
            h = torch.permute(h, (1, 0, 2))
            loss = criterion(h, y, lh, ly)

        total_loss += float(loss)
        vdist += calculate_levenshtein(torch.permute(h, (1, 0, 2)), y, lh, ly, decoder, phoneme_map)

        batch_bar.set_postfix(loss="{:.04f}".format(float(total_loss / (i + 1))), dist="{:.04f}".format(float(vdist / (i + 1))))

        batch_bar.update()

        del x, y, lx, ly, h, lh, loss
        torch.cuda.empty_cache()

    batch_bar.close()
    total_loss = total_loss/len(val_loader)
    val_dist = vdist/len(val_loader)
    return total_loss, val_dist

## Training Setup

In [35]:
checkpoint_folder = '/home/ljc/aq/hw3p2/checkpoint'
os.makedirs(checkpoint_folder, exist_ok=True)
def save_model(model, optimizer, scheduler, metric, epoch, path):
    checkpoint_path = os.path.join(checkpoint_folder, path)
    torch.save(
        {'model_state_dict'         : model.state_dict(),
         'optimizer_state_dict'     : optimizer.state_dict(),
         'scheduler_state_dict'     : scheduler.state_dict(),
         'metric'               : metric[1],
         'epoch'                    : epoch},
        checkpoint_path
    )
    print(f"Checkpoint saved ")


def load_model(model, optimizer=None, scheduler=None, path='best_path.pth'):
    path = os.path.join(checkpoint_folder, path)
    checkpoint = torch.load(path)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    else:
        optimizer = None
    if scheduler is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    else:
        scheduler = None
    epoch = checkpoint['epoch']
    # metric = checkpoint['metric']
  
    return model, optimizer, scheduler, epoch


# Initialize variables for tracking best accuracy and loss
best_valid_loss = 100.0  # To keep track of the best validation accuracy

# Check if there's a checkpoint to resume training from
resume_from_checkpoint = True  # Set this to True if you want to resume from a saved checkpoint
best_model_path = 'best_path.pth'  # The checkpoint filename
if resume_from_checkpoint:
    model, _, _, _ = load_model(model, optimizer=None, scheduler=None, path =best_model_path )
else:
    start_epoch = 0  # Start from scratch if not resuming

In [43]:
# # This is for checkpointing, if you're doing it over multiple sessions

# last_epoch_completed = 0
# start = last_epoch_completed
# end = config["epochs"]
best_lev_dist = float("inf") # if you're restarting from some checkpoint, use what you saw there.
# # epoch_model_path = #TODO set the model path( Optional, you can just store best one. Make sure to make the changes below )
# best_model_path = 'best_model.pth' 

In [42]:
torch.cuda.empty_cache()
gc.collect()

#TODO: Please complete the training loop

for epoch in range(0, config['epochs']):

    print("\nEpoch: {}/{}".format(epoch+1, config['epochs']))

    curr_lr = float(optimizer.param_groups[0]['lr'])

    train_loss              = train_model(model, train_loader,criterion,optimizer)
    valid_loss, valid_dist  = validate_model(model, val_loader, decoder, LABELS)
    scheduler.step(valid_dist)

    print("\tTrain Loss {:.04f}\t Learning Rate {:.07f}".format(train_loss, curr_lr))
    print("\tVal Dist {:.04f}%\t Val Loss {:.04f}".format(valid_dist, valid_loss))


    wandb.log({
        'train_loss': train_loss,
        'valid_dist': valid_dist,
        'valid_loss': valid_loss,
        'lr'        : curr_lr
    })

    save_model(model, optimizer, scheduler, ['valid_dist', valid_dist], epoch, os.path.join(checkpoint_folder, best_model_path))
    # wandb.save(epoch_model_path)
    print("Saved epoch model")

    if valid_dist <= best_lev_dist:
        best_lev_dist = valid_dist
        save_model(model, optimizer, scheduler, ['valid_dist', valid_dist], epoch, os.path.join(checkpoint_folder, best_model_path))
        # wandb.save(best_model_path)
        print("Saved best model")
      # You may find it interesting to exlplore Wandb Artifcats to version your models
run.finish()


Epoch: 1/100


	Train Loss 0.0368	 Learning Rate 0.0002500
	Val Dist 4.3547%	 Val Loss 0.3666
Checkpoint saved 
Saved epoch model
Checkpoint saved 
Saved best model

Epoch: 2/100


	Train Loss 0.0364	 Learning Rate 0.0002500
	Val Dist 4.3786%	 Val Loss 0.3731
Checkpoint saved 
Saved epoch model

Epoch: 3/100


	Train Loss 0.0359	 Learning Rate 0.0002500
	Val Dist 4.3812%	 Val Loss 0.3717
Checkpoint saved 
Saved epoch model

Epoch: 4/100


	Train Loss 0.0351	 Learning Rate 0.0001250
	Val Dist 4.3551%	 Val Loss 0.3726
Checkpoint saved 
Saved epoch model

Epoch: 5/100


...

	Train Loss 0.0339	 Learning Rate 0.0001250
	Val Dist 4.3723%	 Val Loss 0.3757
Checkpoint saved 
Saved epoch model

Epoch: 8/100


	Train Loss 0.0332	 Learning Rate 0.0000625
	Val Dist 4.3428%	 Val Loss 0.3753
Checkpoint saved 
Saved epoch model
Checkpoint saved 
Saved best model

Epoch: 9/100


	Train Loss 0.0331	 Learning Rate 0.0000625
	Val Dist 4.3390%	 Val Loss 0.3753
Checkpoint saved 
Saved epoch model
Checkpoint saved 
Saved best model

Epoch: 10/100


Train:  98%|█████████▊| 110/112 [04:25<00:04,  2.35s/it, loss=0.0327, lr=0.000063]

KeyboardInterrupt: 

In [44]:
best_path = os.path.join(
    checkpoint_folder,
    'best_path.pth'
)
model, _, _, _= load_model(model, optimizer=None, scheduler=None, path =best_model_path )

# Generate Predictions and Submit to Kaggle

In [45]:
#TODO: Make predictions

# Follow the steps below:
# 1. Create a new object for CTCBeamDecoder with larger (why?) number of beams
# 2. Get prediction string by decoding the results of the beam decoder

TEST_BEAM_WIDTH = 40

test_decoder    = CTCBeamDecoder(LABELS, beam_width=TEST_BEAM_WIDTH, log_probs_input=True)
results = []

model.eval()
print("Testing")
for data in tqdm(test_loader):

    x, lx   = data
    x       = x.to(device)

    with torch.no_grad():
        h, lh = model(x, lx)

    prediction_string= decode_prediction(h, lh, test_decoder)# TODO call decode_prediction
    #TODO save the output in results array.
    results.extend(prediction_string)

    del x, lx, h, lh
    torch.cuda.empty_cache()

Testing


100%|██████████| 11/11 [00:29<00:00,  2.72s/it]


In [46]:
data_dir = f"/home/ljc/aq/hw3p2/random_submission.csv"
df = pd.read_csv(data_dir)
df.label = results
df.to_csv('submission.csv', index = False)

In [47]:
# !kaggle competitions submit -c 11-785-hw3p2-f24 -f submission.csv -m "I made it!"
!kaggle competitions submit -c automatic-speech-recognition-asr-slack -f submission.csv -m "I made it"

100%|█████████████████████████████████████████| 210k/210k [00:00<00:00, 694kB/s]
Successfully submitted to Automatic Speech Recognition (ASR) Slack